In [7]:
import numpy as np
import pandas as pd
from pid.utils.generate import generate_cov_from_config, random_rotation_mxy, merge_covs

In [8]:
# Starting from 2-D case
dm, dx, dy = 2, 2, 2
theta = 0
gain_y = np.sqrt(2)
gains = np.linspace(0.5, 3, 10)
gains[3] = 0.99  # A gain of 1 is unstable

In [9]:
for i, gain_x in enumerate(gains):
    cov, *dims = generate_cov_from_config(d=(dm, dx, dy), gain=(gain_x, gain_y), theta=theta)
    # Generate random samples from the multivariate Gaussian distribution
    num_samples = 10  # Number of samples to generate
    mean = np.zeros(sum(dims))
    samples = np.random.multivariate_normal(mean, cov, num_samples)
    print(f'{samples} \n')

[[ 0.10631167 -1.50600129  0.41044458 -1.75740374  1.6460365  -1.19093023]
 [ 2.08921687  0.6549727  -0.64174213 -0.19492278  2.82711956  0.81951641]
 [ 1.63264786  1.6003594   3.02707914  0.41341606  1.76213666  3.94972945]
 [-1.7678379  -0.62513997 -0.63544344  1.05843057 -1.67657369 -2.12902227]
 [-0.41220012 -0.01475603  0.38861579 -0.94873481 -0.22563758  1.15638507]
 [ 1.49410524 -1.59704596  0.6123099  -3.41665654  3.09652462 -1.51581645]
 [ 0.45200976  0.40166127  0.02105094  1.05091176  0.09374608  0.86447806]
 [ 0.13571558 -0.33462623 -0.67243845  0.18493577  1.05726791  0.6903552 ]
 [-0.29112559  0.37354976  0.95641297  1.73032536 -0.5326338   0.19481859]
 [-2.35609894  0.13989861 -1.30034524  0.84473835 -2.45323642 -0.98700986]] 

[[-0.69338373 -0.20653279 -0.52835205  0.55715743  0.10761459 -1.04921722]
 [-1.04452554  0.06379276 -0.91409878  1.50547616 -0.48782219  0.26363239]
 [-0.50970764  0.75019091 -1.4871963   0.90229646 -2.00417186  2.32780233]
 [ 2.13554732  0.33699

In [10]:
# Extend to multi-dimensional case
gains = np.linspace(0, 3, 10)
gains[3] = 0.99  # A gain of 1 is unstable
#gains = np.array([3.0,])
num_doubles = 3
random_rotn = True

# for simplicity, I only print gains[0]
gain = gains[0]
for j in range(num_doubles):
    if j == 0:
        dm, dx, dy = 2, 2, 2
        hx = np.array([[gain, 0], [0, 1]])
        hy = np.array([[1, 0], [0, np.sqrt(2)]])
        sigm = np.eye(dm)
        sigx_m = np.eye(dx)
        sigy_m = np.eye(dy)
        sigw = np.zeros((dx, dy))
        cov = np.block([[sigm, sigm @ hx.T, sigm @ hy.T],
                        [hx @ sigm, hx @ sigm @ hx.T + sigx_m, hx @ sigm @ hy.T + sigw],
                        [hy @ sigm, hy @ sigm @ hx.T + sigw.T, hy @ sigm @ hy.T + sigy_m]])
        cov_old = cov.copy()
    else:
        cov, dm, dx, dy = merge_covs(cov_old, cov_old.copy(), dm, dx, dy)
        cov_old = cov.copy()

    if random_rotn:
        cov = random_rotation_mxy(cov, dm, dx, dy)

    num_samples = 1000  # Number of samples to generate
    mean = np.zeros(sum([dm, dx, dy]))
    samples = np.random.multivariate_normal(mean, cov, num_samples)
    print(f'j: {j}, samples.shape: {samples.shape} \n {samples} \n ')

    dm = dx = dy = 2**(j+1)

j: 0, samples.shape: (1000, 6) 
 [[ 0.57596874  0.38125274  0.16434548 -1.7220108   0.10022338 -0.0961422 ]
 [-0.27933024  0.84592608 -0.38440623  1.29656799 -1.4273831  -1.13812752]
 [-0.16649926  1.41802853 -1.37830885 -0.79825243 -1.17217859 -0.08168552]
 ...
 [-0.8229233  -0.26388787  0.8497299   0.74453326  0.05829922 -0.3579354 ]
 [-0.20613547  1.38193892 -0.63862017 -0.22115681 -1.77849512  1.54412639]
 [-0.56259619 -0.17746573  0.31271237 -0.56113933 -0.55756035  0.2888687 ]] 
 
j: 1, samples.shape: (1000, 12) 
 [[-0.09288666  0.32060319 -0.82918019 ...  0.12882903 -0.260612
  -0.88535985]
 [-0.41971303  1.99520633 -0.69542099 ...  0.07669966  1.79287829
   0.39773197]
 [-1.73969549 -2.17039016 -0.845059   ...  4.26178704 -1.02992577
  -0.78872302]
 ...
 [ 0.19605804 -0.81976264 -0.16871739 ... -0.33164183 -0.40660533
  -0.60687536]
 [-0.78375005  0.6062235   0.61443168 ... -1.95763518  0.24407429
   0.82781138]
 [ 0.11086466 -1.17301934  1.03123181 ...  1.79835422  0.54218624


In [11]:
from pid.optimizers import exact_gauss_tilde_pid, exact_gauss_thin_pid

m = samples[:, :dm]
x = samples[:, dm:dm+dx]
y = samples[:, dm+dx:]
mxy = np.vstack((m, x, y))
cov = np.corrcoef(mxy)    # Shape: (dm+dx+dy, dm+dx+dy)
print(cov)

ret = exact_gauss_tilde_pid(cov, dm, dx, dy)
print(ret[7], ret[5], ret[6], ret[8])

[[ 1.         -0.59899038  0.36183392 ... -0.330469    0.06293336
   0.54104   ]
 [-0.59899038  1.          0.19449531 ... -0.13764567  0.57546697
  -0.14641915]
 [ 0.36183392  0.19449531  1.         ... -0.40763627  0.55408692
  -0.08618645]
 ...
 [-0.330469   -0.13764567 -0.40763627 ...  1.          0.04493184
   0.20799408]
 [ 0.06293336  0.57546697  0.55408692 ...  0.04493184  1.
   0.40853915]
 [ 0.54104    -0.14641915 -0.08618645 ...  0.20799408  0.40853915
   1.        ]]


/Users/adithya/Documents/flow-pid/pid/utils/random_channel.py:8: LinAlgWarning: Ill-conditioned matrix (rcond=5.01248e-17): result may not be accurate.
  return la.solve(a, b, assume_a='pos')


LinAlgError: Matrix is singular.